In [1]:
import sys
sys.path.append('../') # ces deux lignes permettent au notebook de trouver le chemin jusqu'au code source contenu dans 'src'
sys.path.append('../src/') # Yvan dit que c'est absurde, mais sans ça ne marche pas
from src import * # on importe ce qui se trouve dans 'src'
import pandas as pd
from src import script_initialisation, clean_corrupted_data
from src.data import cleaning
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')
import io_utils, data_cleaning
from src.data.constants import INTERIM_DATA_DIR
from src.features import utils
from src.data.variable_constant import FILES_BY_TP # liste de liste de noms de fichiers Thomas : manip + programmation
from src.data.variable_constant import FUNCTIONS_BY_TP # liste de liste de noms de fonctions Thomas : manip + programmation
from src.data.variable_constant import NOMS_TP_PROGRAMMATION # slt les noms de fichiers Programmation - Mirabelle
import re


## Questions

- The index 292469 is Run.Test not Run.program, why you checked it in the CommandRun for the Run.program?

In [2]:
#df.loc[292469]

In [3]:
df = io_utils.reading_dataframe(dir= INTERIM_DATA_DIR, file_name='traces250102_clean.csv')

The file traces250102_clean.csv exists in the directory.


### Replace None values in filename by ""

In [4]:
#Before 
df['filename'].isnull().sum()

151192

In [5]:
df = data_cleaning.replace_None_by_str(df,'filename')

In [6]:
#After
df['filename'].isnull().sum()

0

In [7]:
# Test for empty strings
(df['filename'] == '').sum()

151192

### Split filename, extract the filename

In [8]:
# Before
len(df['filename'].unique())

7090

In [9]:
df['filename'].head(50)

0                                                      
1       /home/l1/abaly.oura.etu/Downloads/puissance4.py
2                                                      
3                                                      
4            /home/l1/abaly.oura.etu/TDM 8/devinette.py
5            /home/l1/abaly.oura.etu/TDM 8/devinette.py
6                                                      
7     /home/l1/abaly.oura.etu/TDM 8/erreurs_boucles_...
8            /home/l1/abaly.oura.etu/TDM 8/devinette.py
9            /home/l1/abaly.oura.etu/TDM 8/devinette.py
10           /home/l1/abaly.oura.etu/TDM 8/devinette.py
11           /home/l1/abaly.oura.etu/TDM 8/devinette.py
12           /home/l1/abaly.oura.etu/TDM 8/devinette.py
13                                                     
14                                                     
15               /home/l1/abaly.oura.etu/TDM 8/while.py
16                                                     
17    /home/l1/abaly.oura.etu/Documents/iterable

In [10]:
df['filename'] = data_cleaning.extract_filename(df['filename'])

In [11]:
# After
df['filename'].head(50)

0                             
1                puissance4.py
2                             
3                             
4                 devinette.py
5                 devinette.py
6                             
7     erreurs_boucles_while.py
8                 devinette.py
9                 devinette.py
10                devinette.py
11                devinette.py
12                devinette.py
13                            
14                            
15                    while.py
16                            
17            iterables_for.py
18                            
19                            
20            iterables_for.py
21      iterable_indexation.py
22                            
23                            
24                            
25                            
26                            
27                            
28                            
29                            
30                            
31                     ytdd.py
32      

## Fill the Nan value of filename column

### Filename column for Run.Test rows

In [12]:
df[df['verb']=='Run.Test']['filename'].unique()

array(['devinette.py', 'ytdd.py', 'iterable_indexation.py',
       'experimentations_fichiers.py', 'puissance4.py',
       'erreurs_boucles_while.py', 'iterables_for.py', 'tictactoe.py',
       'fichiers.py', 'erreurs_aliasing.py',
       'erreurs_boucles_while_suite.py', 'parcours_interrompu.py',
       'erreurs_cond().py', 'conditionnels.py', 'manipulations.py',
       'OURATDM3.py', 'erreurs_cond.py', 'conditionnelles.py',
       'erreurs_cond(1).py', 'categories.py', 'pendu.py', 'fichier.py',
       'while.py', 'imprimerie.py', 'jeu_421.py', 'TP-PROG.py',
       'erreurs_multiples.py', 'booleens.py', 'pour_aller_plus_loin.py',
       'complementaire.py', 'suplémentaire.py', 'age.py',
       'controleTPi.py', 'ab.py', 'abderrahmen ayadi prog.py',
       'programation.py', 'prog.py', 'erreur.py', 'asba.py',
       'fonctions.py', 'exemple_simple.py', 'vzvzfzfzz.py',
       'echappement.py', 'interactions_mystere.py', 'premier_script.py',
       'tuyt.py', 'fonctions .py', 'pour_debog

In [13]:
print(df[df['verb']=='Run.Test']['filename'].isna().sum())
print((df[df['verb']=='Run.Test']['filename'] == '').sum())

0
0


There is no Nan or empty strings, OK for filename of Run.Test

### Fill filename column of rows where their verb column = Run.Program

In [14]:
len(df[df['verb']=='Run.Program'])

54352

In [15]:
#Before
print(df[df['verb']=='Run.Program']['filename'].isna().sum())
print((df[df['verb']=='Run.Program']['filename'] == '').sum())

0
54352


All values of filename for the rows where verb == Run.Program are an empty string.

#### Fill by commandRan

In [16]:
# Apply
mask = df['verb'] == 'Run.Program'
df.loc[mask, 'filename'] = data_cleaning.extract_filename_from_commandRan_Run_Program(df.loc[mask, 'commandRan'])

In [17]:
# After
print(df[df['verb']=='Run.Program']['filename'].isna().sum())
print((df[df['verb']=='Run.Program']['filename'] == '').sum())

0
5791


No none value, the number of empty strings reduced from 54352 to 5791

In [18]:
df[df['verb']=='Run.Program'][['filename','commandRan']].head(50)

,filename,commandRan
22,,%Run -c $EDITOR_CONTENT\n
23,,%Run -c $EDITOR_CONTENT\n
24,,%Run -c $EDITOR_CONTENT\n
25,,%Run -c $EDITOR_CONTENT\n
26,,%Run -c $EDITOR_CONTENT\n
27,,%Run -c $EDITOR_CONTENT\n
28,,%Run -c $EDITOR_CONTENT\n
29,,%Run -c $EDITOR_CONTENT\n
30,,%Run -c $EDITOR_CONTENT\n
45,ytdd.py,%Run ytdd.py\n


#### Fill with P_codeState for those without commandRan

In [19]:
# Apply
mask = (df['verb'] == 'Run.Program') & (df['filename'] == '')
df.loc[mask, 'filename'] = df.loc[mask, 'P_codeState'].map(data_cleaning.extract_filename_from_P_codestate_Run_Program)

In [20]:
# After
print((df[df['verb']=='Run.Program']['filename'] == '').sum())

4623


Empty strings reduced from 5791 to 4623

#### Test for values with different pattern or empty string in filename

In [29]:
pattern = re.compile(r'^[\w\-]+\.py$')
filenames = df[df['verb']=='Run.Program']['filename']
invalid_indices = filenames[~filenames.apply(lambda val: isinstance(val, str) and bool(pattern.fullmatch(val.strip())))].index
invalid_indices, len(invalid_indices)

(Index([   832,    833,    984,   1054,   1055,   1056,   1058,   1322,   1323,
          1324,
        ...
        304614, 304615, 304619, 304620, 306933, 306934, 306935, 306936, 306941,
        306942],
       dtype='int64', length=8266),
 8266)

In [22]:
len(invalid_indices)

8266

In [ ]:
df.loc[1324]['P_codeState'] # Remove ODI?

'# Équipe pédagogique de ODI\n# Ce script réalise un calcul mathématique très compliqué ;)\n# 09/2023\n\nv1 = 3 + 4\nv2 = 5\nres = v1 * v2\n'

In [32]:
df.loc[832]['P_codeState']

'# len("La mer.")\n# max(1, len("abc"), 123+456)\n# len(\'\')\n# len(\'123+456\')\n# len(2)\n# float(\'2.1\')\n# float(\'bof\')\n# 7\n# 579\n# 0\n# 7\n# Traceback (most recent call last):\n#   File "<stdin>", line 5, in <module>\n# TypeError: object of type \'int\' has no len()\n# >>> import math\n# >>> help(math.log)\n# Help on built-in function log in module math:\n# \n# log(...)\n#     log(x, [base=math.e])\n#     Return the logarithm of x to the given base.\n#     \n#     If the base not specified, returns the natural logarithm (base e) of x.\n# La fonction log() du module math calcule le logarithme naturel d\'un nombre\n#\n# >>> résultat1= math.log(100,10)\n# >>> print(résultat1)\n# 2.0\n# >>> résultat2=  math.log(2)\n# >>> print(résultat2)\n# 0.6931471805599453\n#je liste les fonctions du module math avec la fonction dir.\n# >>> math_fonctions=dir(math)\n# >>> print(math_fonctions)\n# [\'__doc__\', \'__loader__\', \'__name__\', \'__package__\', \'__spec__\', \'acos\', \'acosh\', 

In [37]:
df.loc[invalid_indices]['P_codeState']

832       # len("La mer.")\n# max(1, len("abc"), 123+456...
833       # len("La mer.")\n# max(1, len("abc"), 123+456...
984       # L1MI-S1 - Prog - semaine 4 : conditionnelles...
1054                                                    NaN
1055                                                    NaN
                                ...                        
306934                  prenom = "Noe"\nlogiciel = "VSCODE"
306935                                                  NaN
306936    # Équipe pédagogique de ODI\n# Ce script réali...
306941    # Équipe pédagogique de ODI\n# Ce script réali...
306942    # Équipe pédagogique de ODI\n# Ce script réali...
Name: P_codeState, Length: 8266, dtype: object

In [36]:
df.loc[306936]['P_codeState']

'# Équipe pédagogique de ODI\n# Ce script réalise un calcul mathématique très compliqué ;)\n# 09/2023\n\nv1 = 3 + 4\nv2 = 5\nres = v1 * v2\n'

In [26]:
df.loc[292501] # What should I do with this?

research_usage                                                          1.0
result                                                                  NaN
_id.$oid                                           66d97bc1bd5a98b8f9d93d07
timestamp.$date                            2024-09-05 11:36:55.590000+00:00
stored.$date                                       2024-08-29T07:34:11.687Z
actor                                                   waylan.godfrey.etu/
actor.objectType                                                      Agent
verb                                                            Run.Program
object                    https://www.cristal.univ-lille.fr/objects/Program
object.objectType                                                  Activity
object.extension                                                        NaN
session.id                                                  140568420679440
commandRan                                                           %Run\n
result.succe

In [27]:
df.loc[292469] # don't care because verb = Run.Test

research_usage                                                          1.0
result                                                                  NaN
_id.$oid                                           66f67bebbd5a98b8f9dab7fd
timestamp.$date                            2024-09-27 11:33:10.430000+00:00
stored.$date                                       2024-08-29T07:34:11.687Z
actor                                                   waylan.godfrey.etu/
actor.objectType                                                      Agent
verb                                                               Run.Test
object                      https://www.cristal.univ-lille.fr/objects/Tests
object.objectType                                                  Activity
object.extension                                                        NaN
session.id                                                  140568162309344
commandRan                                                              NaN
result.succe